### Import Dependencies

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os
env_path = Path("../../.env").resolve()
loaded = load_dotenv(env_path, override=True)  # override=True replaces stale empty values
print("load_dotenv:", loaded)
print("env path:", env_path)

import openai
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree

load_dotenv: True
env path: /Users/bentonturner/Documents/GitHub/ai-engineering-bootcamp-cohort-5/.env


In [2]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

### Embedding function

In [3]:
@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "text-embedding-3-small"
    }
)
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens,
        }

    return response.data[0].embedding

### Retrieval function

In [4]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [5]:
@traceable(
    name="retrieve_data",
    run_type="retriever"
)
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

### Format retrieved context function

In [6]:
@traceable(
    name="format_retrieved_context",
    run_type="prompt"
)
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

### Create prompt template function

In [8]:
@traceable(
    name="build_prompt",
    run_type="prompt"
)
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

### Generate answer function

In [17]:
@traceable(
    name="generate_answer",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-nano"
    }
)
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
        }

    return response.choices[0].message.content

### Combined RAG pipeline

In [18]:
@traceable(
    name="rag_pipeline",
)
def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer
    

In [13]:
print(rag_pipeline("Do you have a USB connectable fan for hot summers?"))

I don’t have any USB connectable fans in the available products list. The items currently available are music and entertainment products (for example: Radiate Like This, Sheer Heart Attack, and The Complete Harry Potter Film Music Collection).


In [14]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have above 4 rating.", 10))

Based on the available products with a rating above 4, here are the top picks (all are rated 4.7 or 4.8):

- Against The Odds: 1974-1982 [8 CD] (4.8)
- The Complete Harry Potter Film Music Collection (4.8)
- All My Friends: Celebrating The Songs & Voice Of Gregg Allman [4 LP] (4.8)
- Bluey Dance Mode Orange (4.8)
- 5-STAR VER. A (4.8)
- Beatopia[LP] (4.8)
- Songs About You (4.8)
- Girls, Girls, Girls (4.8)
- Radiate Like This[LP] (4.7)

Note: The available items listed here are albums/collections, not earphones. If you want, tell me whether you’re looking for wired or wireless earphones and any budget, and I’ll narrow it down if those are available.


In [15]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have bellow 4 rating.", 10))

I can’t recommend earphones from the available items because the available products shown here are all music/music collections (LPs, CDs, film music collections, etc.), not earphones—and the only ratings shown are 4.6 to 4.8, none below 4.

If you share the type of earphones you want (wired or wireless) or ask again with a list that includes earphones, I can suggest options with rating below 4.
